##  Solution for Some subtask
resposible: Ivett

In [1]:
# load sample graph

from build_graphs import to_algs4, parse_graph
from itu.algs4.graphs.breadth_first_paths import BreadthFirstPaths
from itu.algs4.graphs.topological import Topological
from pathlib import Path
import networkx as nx

# filename = "G-ex"
filename = "ski-illustration"

path = f"../red-scare/data/{filename}.txt"

In [2]:
adj_list, red_vertices, s, t, directed = parse_graph(path=path)

In [3]:
G, index, names = to_algs4(adj_list, directed=directed)

In [4]:
def some_dag(G, index, red_vertices, s, t):
    # check everything reachable from s 
    from_s = BreadthFirstPaths(G, index[s])
    #check everything what reachables t
    to_t = BreadthFirstPaths(G.reverse(), index[t])
    #returning any path from s to t that goes through a red vertex
    return any(from_s.has_path_to(index[r]) and to_t.has_path_to(index[r])for r in red_vertices)
        

In [5]:
def some_undirected(G, index, red_vertices, s, t):
    # converting algs4 graph to networkx graph
    G_nx = nx.Graph()
    for v in range(G.V()):
        for w in G.adj(v):
            G_nx.add_edge(v, w)

    # add temporary edge between s-t, so everything is one block
    G_nx.add_edge(index[s], index[t])

    for block in nx.biconnected_components(G_nx):
        if index[s] in block and index[t] in block:
            # check if any red vertex is in the same block as s and t
            return any(index[r] in block for r in red_vertices)

    return False

In [6]:
def some(G, index, red_vertices, s, t, directed):
    # if directed, check if DAG  or not
    if directed:
        if Topological(G).has_order():
            return some_dag(G, index, red_vertices, s, t)
        return "Are you sure bruw?"
        
    # check first if s-t path is possible:
    has_path = BreadthFirstPaths(G, index[s]).has_path_to(index[t])
    if not has_path:
        return False
    # if s or t is red and there's a path from s to t, return True
    if s in red_vertices or t in red_vertices:
        return True
    #no reds -> false
    if not red_vertices:
        return False
    
    #if undirected, fall back to some_undirected
    return some_undirected(G, index, red_vertices, s, t)

In [7]:
some(G, index, red_vertices, s, t, directed)

True

In [8]:
data_dir = Path("../red-scare/data")
results = {}

for fp in sorted(data_dir.glob("*.txt")):
    adj, reds, s_, t_, d = parse_graph(fp)
    G_, index_, names_ = to_algs4(adj, d)
    results[fp.stem] = some(G_, index_, reds, s_, t_, d)

out = Path("../red-scare/output")
out.mkdir(parents=True, exist_ok=True)
with open(out / "some_answers.txt", "w") as f:
    for name, ans in results.items():
        f.write(f"{name} : {ans}\n")

print(len(results), "instances written")

154 instances written


In [9]:
#rewriting the code with networkx as noone else used alsg4

def some_nx(adj_list, red_vertices, s, t, directed):
    #build graphs with networkx
    G = nx.DiGraph() if directed else nx.Graph()
    G.add_nodes_from(adj_list)
    G.add_edges_from((u, v) for u in adj_list for v in adj_list[u] if u != v)


    # if directed, check if DAG  or not
    if directed:
        if not nx.is_directed_acyclic_graph(G):
            return "?!"
        # check everything reachable from s
        from_s = nx.descendants(G,s) | {s}
        # check everything that can reach t
        to_t = nx.ancestors(G,t) | {t}
        #returning any path from s to t that goes through a red vertex
        return any(r in from_s and r in to_t for r in red_vertices)

    # check first if s-t path is possible:
    if not nx.has_path(G, s, t):
        return False

    # add temporary edge between s-t, so everything is one block
    G.add_edge(s, t)
    for block in nx.biconnected_component(G):
        if s in block and t in block:
            # check if any red vertex is in the same block as s and t
            return any(r in block for r in red_vertices)

    return False